In [0]:
%sql
SELECT * FROM ECOMMERCE.BRONZE.inventory LIMIT 10;

## Loading libraries for future usage

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col


In [0]:
inventory_df = (
    spark.read.table("ECOMMERCE.BRONZE.INVENTORY")
    .withColumn("available_quantity", col("available_quantity").cast("decimal(18,2)"))
    .withColumn("reserved_quantity", col("reserved_quantity").cast("decimal(18,2)"))
    .withColumn("reorder_level", col("reorder_level").cast("decimal(18,2)"))
    .dropDuplicates()
)

## Transformation 1

available_to_sell = available_quantity - reserved_quantity

In [0]:
inventory_df = inventory_df.withColumn('available_to_sell',col('available_quantity')-col('reserved_quantity'))

## Transformation 2
Then classify: 
* available_to_sell <= 0  → OUT_OF_STOCK
* available_to_sell <= reorder_level → LOW_STOCK
* otherwise → IN_STOCK

In [0]:
inventory_df = inventory_df.withColumn(
    "inventory_type",
    when(col("available_to_sell") <= 0, "OUT_OF_STOCK")
    .when(col("available_to_sell") <= col("reorder_level"), "LOW_STOCK")
    .otherwise("IN_STOCK"),
)

In [0]:
inventory_df.write.format("delta").mode("append").saveAsTable(
    "ecommerce.silver.inventory"
)

In [0]:
%sql
SELECT * FROM ECOMMERCE.SILVER.INVENTORY LIMIT 10;